# Лекция 3. Свёрточные нейронные сети (CNN)

## Практический кейс: обнаружение плесени на семенах пшеницы

**Датасет:** GrainSpace (подмножество пшеницы)

**Задача:** бинарная классификация — NORMAL vs MOULDY

**Требования:**
- Точность ≥ 92%
- Скорость ≤ 0.5 сек на зерно
- Объяснимость (Grad-CAM)

## 1. Установка и импорты

In [ ]:
# Установка зависимостей (для Colab)
# !pip install torch torchvision matplotlib numpy scikit-learn grad-cam opencv-python

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import datasets, transforms, models
import torch.nn.functional as F

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

import os
import random
from PIL import Image

# Фиксируем seed для воспроизводимости
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# Устройство
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Используется устройство: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Память: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. Демонстрация: почему MLP не работает с изображениями

In [ ]:
# Считаем параметры MLP для изображения 256×256×3
H, W, C = 256, 256, 3
n_inputs = H * W * C
n_neurons_first_layer = 256

n_weights = n_inputs * n_neurons_first_layer
memory_gb = (n_weights + n_neurons_first_layer) * 4 / 1e9

print('=' * 50)
print('ПРОБЛЕМА MLP ДЛЯ ИЗОБРАЖЕНИЙ')
print('=' * 50)
print(f'Размер изображения: {H}×{W}×{C}')
print(f'Входов: {n_inputs:,}')
print(f'Нейронов в первом слое: {n_neurons_first_layer}')
print(f'Весов в первом слое: {n_weights:,}')
print(f'Память только для весов: {memory_gb:.2f} GB')
print()
print('Сравнение с CNN:')
print(f'  MLP первый слой: {n_weights:,} параметров')
print(f'  CNN слой (32 фильтра 3×3×3): {32 * 3 * 3 * 3 + 32} параметров')
print(f'  Разница: {n_weights / (32 * 3 * 3 * 3 + 32):,.0f} раз')

## 3. Демонстрация свёртки вручную

In [ ]:
# Фрагмент фото зерна (яркость)
image_patch = np.array([
    [180, 182, 185, 183, 180],
    [178, 180, 183, 185, 182],
    [175, 178, 180, 183, 185],
    [172, 175, 178, 180, 183],
    [170, 172, 175, 178, 180]
])

# Фильтр: детектор горизонтальной границы
filter_horizontal = np.array([
    [-1, -1, -1],
    [ 0,  0,  0],
    [ 1,  1,  1]
])

def conv2d_manual(image, kernel):
    """Ручная свёртка без padding, stride=1."""
    H, W = image.shape
    K = kernel.shape[0]
    H_out = H - K + 1
    W_out = W - K + 1
    output = np.zeros((H_out, W_out))
    
    for i in range(H_out):
        for j in range(W_out):
            patch = image[i:i+K, j:j+K]
            output[i, j] = np.sum(patch * kernel)
    
    return output

result = conv2d_manual(image_patch, filter_horizontal)

print('Исходный фрагмент:')
print(image_patch)
print()
print('Фильтр:')
print(filter_horizontal)
print()
print('Feature map:')
print(result)

In [ ]:
# Визуализация
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(image_patch, cmap='gray')
axes[0].set_title('Исходный фрагмент')
axes[0].axis('off')

axes[1].imshow(filter_horizontal, cmap='RdBu', vmin=-1, vmax=1)
axes[1].set_title('Фильтр (детектор границы)')
axes[1].axis('off')

axes[2].imshow(result, cmap='viridis')
axes[2].set_title('Feature map')
axes[2].axis('off')

plt.tight_layout()
plt.show()

## 4. Padding, Stride, Pooling — демонстрация

In [ ]:
def conv2d_advanced(image, kernel, padding=0, stride=1):
    """Свёртка с padding и stride."""
    # Padding
    if padding > 0:
        image = np.pad(image, padding, mode='constant', constant_values=0)
    
    H, W = image.shape
    K = kernel.shape[0]
    H_out = (H - K) // stride + 1
    W_out = (W - K) // stride + 1
    output = np.zeros((H_out, W_out))
    
    for i in range(H_out):
        for j in range(W_out):
            patch = image[i*stride:i*stride+K, j*stride:j*stride+K]
            output[i, j] = np.sum(patch * kernel)
    
    return output

# Тестируем разные padding и stride
test_image = np.random.randn(8, 8)
test_kernel = np.array([[-1, -1, -1], [0, 0, 0], [1, 1, 1]])

print('Вход: 8×8, фильтр 3×3')
print()
print(f'{"Padding":<10} {"Stride":<10} {"Выход":<10}')
print('-' * 30)
for p in [0, 1]:
    for s in [1, 2]:
        out = conv2d_advanced(test_image, test_kernel, padding=p, stride=s)
        print(f'{p:<10} {s:<10} {out.shape}')

In [ ]:
# Max Pooling и Average Pooling вручную
def max_pool2d(x, kernel_size=2, stride=2):
    H, W = x.shape
    H_out = (H - kernel_size) // stride + 1
    W_out = (W - kernel_size) // stride + 1
    output = np.zeros((H_out, W_out))
    for i in range(H_out):
        for j in range(W_out):
            patch = x[i*stride:i*stride+kernel_size, j*stride:j*stride+kernel_size]
            output[i, j] = np.max(patch)
    return output

def avg_pool2d(x, kernel_size=2, stride=2):
    H, W = x.shape
    H_out = (H - kernel_size) // stride + 1
    W_out = (W - kernel_size) // stride + 1
    output = np.zeros((H_out, W_out))
    for i in range(H_out):
        for j in range(W_out):
            patch = x[i*stride:i*stride+kernel_size, j*stride:j*stride+kernel_size]
            output[i, j] = np.mean(patch)
    return output

test = np.array([
    [1, 3, 2, 4],
    [5, 6, 7, 8],
    [9, 2, 11, 12],
    [4, 9, 10, 3]
])

print('Вход:')
print(test)
print()
print('MaxPool 2×2:')
print(max_pool2d(test))
print()
print('AvgPool 2×2:')
print(avg_pool2d(test))

## 5. Загрузка датасета

### Вариант A: GrainSpace (если скачан)
### Вариант B: MNIST (для демонстрации)

In [ ]:
# === ВАРИАНТ A: GrainSpace ===
# Раскомментируйте, если скачали датасет

# from torchvision.datasets import ImageFolder
#
# train_transform = transforms.Compose([
#     transforms.Resize((256, 256)),
#     transforms.RandomRotation(360),
#     transforms.RandomHorizontalFlip(),
#     transforms.RandomVerticalFlip(),
#     transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
#     transforms.ColorJitter(brightness=0.2, contrast=0.2),
#     transforms.ToTensor(),
#     transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
# ])
#
# val_transform = transforms.Compose([
#     transforms.Resize((256, 256)),
#     transforms.CenterCrop(224),
#     transforms.ToTensor(),
#     transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
# ])
#
# train_dataset = ImageFolder('data/grainspace/train', transform=train_transform)
# val_dataset = ImageFolder('data/grainspace/val', transform=val_transform)

In [ ]:
# === ВАРИАНТ B: MNIST (демонстрация) ===
# Используем MNIST как fallback, потому что логика обучения идентична

train_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.RandomRotation(10),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

val_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = datasets.MNIST('./data', train=True, download=True, transform=train_transform)
val_dataset = datasets.MNIST('./data', train=False, download=True, transform=val_transform)

print(f'Train: {len(train_dataset):,} примеров')
print(f'Val: {len(val_dataset):,} примеров')

In [ ]:
# Визуализация примеров
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    img, label = train_dataset[i * 1000]
    # Денормализация
    img_show = img.squeeze() * 0.5 + 0.5
    ax.imshow(img_show, cmap='gray')
    ax.set_title(f'Класс: {label}')
    ax.axis('off')
plt.suptitle('Примеры из датасета')
plt.tight_layout()
plt.show()

## 6. DataLoaders

In [ ]:
batch_size = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size * 2,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(f'Train батчей: {len(train_loader)}')
print(f'Val батчей: {len(val_loader)}')

# Проверим размер батча
X_sample, y_sample = next(iter(train_loader))
print(f'Размер батча: {X_sample.shape}')  # [64, 1, 64, 64]

## 7. Простая CNN с нуля

### Архитектура: Conv → BN → ReLU → Pool

In [ ]:
def conv_block(in_channels, out_channels):
    """Conv → BN → ReLU → Pool"""
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(kernel_size=2, stride=2)
    )


class SimpleCNN(nn.Module):
    """Простая CNN для классификации."""
    
    def __init__(self, num_classes=10):
        super().__init__()
        
        self.features = nn.Sequential(
            conv_block(1, 32),     # [1, 64, 64] → [32, 32, 32]
            conv_block(32, 64),    # → [64, 16, 16]
            conv_block(64, 128),   # → [128, 8, 8]
            conv_block(128, 256),  # → [256, 4, 4]
        )
        
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),   # [256, 4, 4] → [256, 1, 1]
            nn.Flatten(),               # → [256]
            nn.Dropout(0.5),
            nn.Linear(256, num_classes) # → [num_classes]
        )
    
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


model = SimpleCNN(num_classes=10).to(device)

# Считаем параметры
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'Всего параметров: {total_params:,}')
print(f'Обучаемых параметров: {trainable_params:,}')
print()
print(model)

In [ ]:
# Проверка forward pass
with torch.no_grad():
    X_test = torch.randn(2, 1, 64, 64).to(device)
    y_test = model(X_test)
    print(f'Вход: {X_test.shape}')
    print(f'Выход: {y_test.shape}')

## 8. Обучение: полный цикл

In [ ]:
# Гиперпараметры
EPOCHS = 10
LR = 1e-3
WEIGHT_DECAY = 1e-4

# Функция потерь и оптимизатор
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

print('Оптимизатор: AdamW')
print(f'Learning rate: {LR}')
print(f'Weight decay: {WEIGHT_DECAY}')

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    """Одна эпоха обучения."""
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        
        optimizer.zero_grad()
        logits = model(X)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * X.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        total += X.size(0)
    
    return total_loss / total, correct / total


@torch.no_grad()
def validate(model, loader, criterion, device):
    """Валидация."""
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        logits = model(X)
        loss = criterion(logits, y)
        
        total_loss += loss.item() * X.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        total += X.size(0)
    
    return total_loss / total, correct / total

In [ ]:
# Цикл обучения
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0

print('=' * 70)
print(f'{"Epoch":<8}{"Train Loss":<15}{"Train Acc":<15}{"Val Loss":<15}{"Val Acc":<15}')
print('=' * 70)

for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = validate(model, val_loader, criterion, device)
    scheduler.step()
    
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_simple_cnn.pt')
    
    print(f'{epoch+1:<8}{train_loss:<15.4f}{train_acc:<15.4f}{val_loss:<15.4f}{val_acc:<15.4f}')

print('=' * 70)
print(f'Лучшая Val Accuracy: {best_val_acc:.4f}')

In [ ]:
# Графики обучения
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

epochs_range = range(1, EPOCHS + 1)

axes[0].plot(epochs_range, history['train_loss'], 'o-', label='Train')
axes[0].plot(epochs_range, history['val_loss'], 's-', label='Val')
axes[0].set_xlabel('Эпоха')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, history['train_acc'], 'o-', label='Train')
axes[1].plot(epochs_range, history['val_acc'], 's-', label='Val')
axes[1].set_xlabel('Эпоха')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Transfer Learning с ResNet-18

### Используем предобученную модель и дообучаем на нашей задаче

In [ ]:
# Загружаем предобученную ResNet-18
resnet = models.resnet18(weights='IMAGENET1K_V1')

# Заменяем первый слой (ImageNet — RGB, MNIST — grayscale)
resnet.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)

# Заменяем последний слой
resnet.fc = nn.Linear(512, 10)

resnet = resnet.to(device)

total_params = sum(p.numel() for p in resnet.parameters())
print(f'Параметров в ResNet-18: {total_params:,}')

# Проверка forward pass
with torch.no_grad():
    X_test = torch.randn(2, 1, 64, 64).to(device)
    y_test = resnet(X_test)
    print(f'Вход: {X_test.shape} → Выход: {y_test.shape}')

In [ ]:
# === Стратегия 1: Feature Extraction ===
# Замораживаем всё, кроме последнего слоя

for param in resnet.parameters():
    param.requires_grad = False

for param in resnet.fc.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in resnet.parameters() if p.requires_grad)
print(f'Обучаемых параметров (feature extraction): {trainable:,}')

In [ ]:
# === Стратегия 2: Fine-tuning ===
# Размораживаем ВСЕ параметры

for param in resnet.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in resnet.parameters() if p.requires_grad)
print(f'Обучаемых параметров (fine-tuning): {trainable:,}')

In [ ]:
# === Обучение ResNet ===
EPOCHS_RESNET = 5
LR_RESNET = 1e-4  # маленький LR для fine-tuning

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(resnet.parameters(), lr=LR_RESNET, weight_decay=1e-2)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_RESNET)

history_resnet = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0

print('=' * 70)
print('Обучение ResNet-18 (fine-tuning)')
print('=' * 70)
print(f'{"Epoch":<8}{"Train Loss":<15}{"Train Acc":<15}{"Val Loss":<15}{"Val Acc":<15}')
print('-' * 70)

for epoch in range(EPOCHS_RESNET):
    train_loss, train_acc = train_one_epoch(resnet, train_loader, criterion, optimizer, device)
    val_loss, val_acc = validate(resnet, val_loader, criterion, device)
    scheduler.step()
    
    history_resnet['train_loss'].append(train_loss)
    history_resnet['train_acc'].append(train_acc)
    history_resnet['val_loss'].append(val_loss)
    history_resnet['val_acc'].append(val_acc)
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(resnet.state_dict(), 'best_resnet.pt')
    
    print(f'{epoch+1:<8}{train_loss:<15.4f}{train_acc:<15.4f}{val_loss:<15.4f}{val_acc:<15.4f}')

print('-' * 70)
print(f'Лучшая Val Accuracy: {best_val_acc:.4f}')

## 10. Визуализация фильтров первого слоя

In [ ]:
# Достаём фильтры первого свёрточного слоя
weights = model.features[0][0].weight.data.cpu()  # [32, 1, 3, 3]

fig, axes = plt.subplots(4, 8, figsize=(16, 8))
for i, ax in enumerate(axes.flat):
    if i < weights.shape[0]:
        w = weights[i, 0].numpy()
        w_norm = (w - w.min()) / (w.max() - w.min() + 1e-8)
        ax.imshow(w_norm, cmap='gray')
        ax.set_title(f'F{i}', fontsize=8)
    ax.axis('off')

plt.suptitle('Фильтры первого слоя SimpleCNN', fontsize=14)
plt.tight_layout()
plt.show()

## 11. Визуализация feature maps

In [ ]:
# Регистрируем hook для захвата активаций
activations = {}

def get_activation(name):
    def hook(model, input, output):
        activations[name] = output.detach()
    return hook

# Регистрируем на первом conv-слое
model.features[0][0].register_forward_hook(get_activation('conv1'))

# Прогоняем одно изображение
model.eval()
img, label = val_dataset[0]
with torch.no_grad():
    _ = model(img.unsqueeze(0).to(device))

# Визуализируем первые 16 feature maps
act = activations['conv1'][0].cpu()  # [32, H, W]

fig, axes = plt.subplots(4, 4, figsize=(12, 12))
for i, ax in enumerate(axes.flat):
    ax.imshow(act[i].numpy(), cmap='viridis')
    ax.set_title(f'Feature map {i}')
    ax.axis('off')

plt.suptitle(f'Feature maps после первого Conv (класс: {label})', fontsize=14)
plt.tight_layout()
plt.show()

## 12. Grad-CAM: где модель «смотрит»

In [ ]:
# Простая реализация Grad-CAM
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        self._register_hooks()
    
    def _register_hooks(self):
        def forward_hook(module, input, output):
            self.activations = output.detach()
        
        def backward_hook(module, grad_input, grad_output):
            self.gradients = grad_output[0].detach()
        
        self.target_layer.register_forward_hook(forward_hook)
        self.target_layer.register_backward_hook(backward_hook)
    
    def generate(self, input_tensor, target_class=None):
        self.model.eval()
        output = self.model(input_tensor)
        
        if target_class is None:
            target_class = output.argmax(1).item()
        
        self.model.zero_grad()
        score = output[0, target_class]
        score.backward()
        
        # Веса = средние градиенты по пространству
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)  # [1, C, 1, 1]
        
        # Взвешенная сумма активаций
        cam = (weights * self.activations).sum(dim=1, keepdim=True)  # [1, 1, H, W]
        cam = F.relu(cam)
        
        # Нормализация
        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)
        
        return cam[0, 0].cpu().numpy()


# Применяем Grad-CAM на последнем свёрточном слое
target_layer = model.features[3][0]  # последний Conv
grad_cam = GradCAM(model, target_layer)

# Выбираем пример
img, label = val_dataset[0]
input_tensor = img.unsqueeze(0).to(device)

cam = grad_cam.generate(input_tensor, target_class=label)

# Визуализация
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

img_show = img.squeeze().numpy() * 0.5 + 0.5
axes[0].imshow(img_show, cmap='gray')
axes[0].set_title(f'Оригинал (класс: {label})')
axes[0].axis('off')

axes[1].imshow(cam, cmap='jet')
axes[1].set_title('Grad-CAM')
axes[1].axis('off')

# Наложение
import matplotlib.cm as cm
cam_resized = np.array(Image.fromarray(cam).resize(img_show.shape[::-1]))
heatmap = cm.jet(cam_resized)[:, :, :3]
overlay = 0.5 * img_show[..., None] + 0.5 * heatmap
overlay = np.clip(overlay, 0, 1)
axes[2].imshow(overlay)
axes[2].set_title('Наложение')
axes[2].axis('off')

plt.tight_layout()
plt.show()

## 13. Data Augmentation: визуализация

In [ ]:
# Загружаем одно изображение
img_orig, label = datasets.MNIST('./data', train=True, download=True, transform=transforms.ToTensor())[0]

# Набор аугментаций
augmentations = [
    ('Оригинал', transforms.Compose([transforms.ToTensor()])),
    ('Поворот +15°', transforms.Compose([transforms.RandomRotation((15, 15)), transforms.ToTensor()])),
    ('Поворот -15°', transforms.Compose([transforms.RandomRotation((-15, -15)), transforms.ToTensor()])),
    ('Сдвиг', transforms.Compose([transforms.RandomAffine(0, translate=(0.15, 0.15)), transforms.ToTensor()])),
    ('Масштаб 0.8', transforms.Compose([transforms.Resize(22), transforms.Pad(3), transforms.ToTensor()])),
    ('Яркость +', transforms.Compose([transforms.ToTensor(), transforms.Lambda(lambda x: x + 0.3)])),
    ('Яркость -', transforms.Compose([transforms.ToTensor(), transforms.Lambda(lambda x: x - 0.3)])),
    ('Шум', transforms.Compose([transforms.ToTensor(), transforms.Lambda(lambda x: x + 0.2 * torch.randn_like(x))])),
]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

img_pil = transforms.ToPILImage()(img_orig)

for i, (name, aug) in enumerate(augmentations):
    ax = axes.flat[i]
    if name == 'Оригинал':
        img_aug = img_orig
    else:
        img_aug = aug(img_pil)
    ax.imshow(img_aug.squeeze(), cmap='gray')
    ax.set_title(name)
    ax.axis('off')

plt.suptitle('Data Augmentation: одно изображение — разные варианты', fontsize=14)
plt.tight_layout()
plt.show()

## 14. Финальная оценка модели

In [ ]:
# Загружаем лучшую модель
model.load_state_dict(torch.load('best_simple_cnn.pt'))
model.eval()

# Собираем предсказания
all_preds = []
all_labels = []

with torch.no_grad():
    for X, y in val_loader:
        X = X.to(device)
        logits = model(X)
        preds = logits.argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(y.numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# Метрики
print('=' * 60)
print('ФИНАЛЬНАЯ ОЦЕНКА МОДЕЛИ')
print('=' * 60)
print(classification_report(all_labels, all_preds))

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
print('Confusion Matrix:')
print(cm)

In [ ]:
# Визуализация confusion matrix
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(10))
ax.set_yticks(range(10))
ax.set_xlabel('Предсказание')
ax.set_ylabel('Истина')
ax.set_title('Confusion Matrix')

for i in range(10):
    for j in range(10):
        ax.text(j, i, cm[i, j], ha='center', va='center',
                color='white' if cm[i, j] > cm.max() / 2 else 'black')

plt.colorbar(im)
plt.tight_layout()
plt.show()

## 15. Выводы

### Что мы сделали
1. **Показали проблему MLP** для изображений — 50M параметров в первом слое.
2. **Разобрали свёртку** — фильтры, feature maps, padding, stride, pooling.
3. **Собрали CNN** — Conv → BN → ReLU → Pool → AdaptiveAvgPool → FC.
4. **Обучили с нуля** и **через transfer learning** (ResNet-18).
5. **Визуализировали** фильтры, feature maps, Grad-CAM.
6. **Применили аугментацию** и **метрики**.

### Ключевые идеи
- **Разделение весов** — один фильтр на всё изображение.
- **Локальные связи** — нейрон смотрит только на область 3×3.
- **Инвариантность к сдвигу** — pooling и stride.
- **Иерархия признаков** — от краёв к целым объектам.
- **Transfer learning** — предобученные модели спасают при малых данных.

### Для GrainSpace (реальная задача)
- Заменить MNIST на GrainSpace.
- Использовать `pos_weight=3.0` из-за дисбаланса.
- Добиться AUC > 0.95.
- Проверить Grad-CAM: смотрит ли модель на плесень, а не на фон.